# Лабораторная работа № 1
## Разведочный анализ больших данных с использованием фреймворка Apache Spark
### Часть 1. Загрузка данных в таблицу Apache Iceberg

**Датасет:** [US Accidents (2016 – 2023)](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) — 7,7 млн дорожно-транспортных происшествий в 49 штатах США, один CSV-файл на 2,8 ГБ.

**Цель части:** перенести сырой CSV из папки проекта в `HDFS`, а оттуда — в таблицу `Apache Iceberg` с правильными именами и типами столбцов. Разведочный анализ — во второй части.

### Оглавление

1. [Проверка кластера](#ch1)
2. [Запуск Spark-сессии](#ch2)
3. [Загрузка CSV в HDFS](#ch3)
4. [Описание признаков](#ch4)
5. [Имена столбцов](#ch5)
6. [Типы данных](#ch6)
7. [Сохранение в Iceberg](#ch7)
8. [Выводы](#ch8)

<a id="ch1"></a>
## 1. Проверка кластера

**Путь:** проверяем, что `HDFS` и `YARN` работают. Без этого Spark-сессия не запустится.

In [1]:
!hdfs dfs -df -h

Filesystem                   Size    Used  Available  Use%
hdfs://hadoop-node:9000  1006.9 G  10.8 G    875.8 G    1%


Свободного места в `HDFS` достаточно. Вторая проверка — узлы `YARN`, на которых будут работать исполнители Spark.

In [2]:
!yarn node -list 2>/dev/null

Total Nodes:1
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
hadoop-node:34009	        RUNNING	 hadoop-node:8042	                           0


Узлы в состоянии `RUNNING` — кластер готов.

<a id="ch2"></a>
## 2. Запуск Spark-сессии

**Путь:** создаём сессию под управлением `YARN` и подключаем каталог `Iceberg`, в котором будут храниться таблицы.

**Пути.** Всё, что нужно работе, лежит внутри проекта `SOBD-labs`: исходный CSV — в `SOBD/2024/data/`, рисунки — в `SOBD/2024/figures/`. В `HDFS` проект занимает одну папку `/user/<логин>/SOBD-labs/` с данными, таблицами и моделью.

**Ресурсы** выбираются по машине. Узел кластера кафедры (256 ядер, 188 ГБ) делят много студентов, поэтому берём около 12 %: 4 исполнителя по 8 ядер и 6 ГБ. На локальном стенде с 8 ГБ памяти — 2 исполнителя по 2 ядра и 1,5 ГБ. После перемешивания (shuffle) Spark по умолчанию делит данные на 200 разделов; ставим по 4 раздела на ядро — лишние мелкие задачи только добавляют накладные расходы.

In [3]:
import os
import re
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

user = os.getenv("USER")

# Все пути — от папки проекта SOBD-labs: ноутбук лежит в SOBD/2024/lab1 или lab2,
# а в HDFS проект занимает папку SOBD-labs домашнего каталога (data/, warehouse/, models/)
BASE = Path.cwd().parent
HDFS_ROOT = f"hdfs:///user/{user}/SOBD-labs"

# Ресурсы. Узел кафедры (256 ядер, 188 ГБ) делят много студентов — берём ~12 %;
# на локальном стенде (4 ядра, 8 ГБ) — то, что помещается в его YARN
on_cluster = os.cpu_count() >= 64
executors, cores, memory, driver_memory = (4, 8, "6g", "4g") if on_cluster else (2, 2, "1536m", "2g")
# библиотека Iceberg: на кластере скачивается из Maven, на стенде уже лежит среди jar-файлов Spark
iceberg_package = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0" if on_cluster else ""

spark = (
    SparkSession.builder
    .appName("lab1-part1-loading")
    .master("yarn")
    .config("spark.dynamicAllocation.enabled", "false")  # ровно столько исполнителей, сколько задали
    .config("spark.executor.instances", executors)
    .config("spark.executor.cores", cores)
    .config("spark.executor.memory", memory)
    .config("spark.driver.memory", driver_memory)
    .config("spark.sql.shuffle.partitions", 4 * executors * cores)  # 4 раздела на ядро вместо 200
    # Iceberg: таблицы и их метаданные — в HDFS
    .config("spark.jars.packages", iceberg_package)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.spark_catalog.type", "hadoop")
    .config("spark.sql.catalog.spark_catalog.warehouse", f"{HDFS_ROOT}/warehouse")
    # без кеша каталога: после перезаписи таблицы сессия сразу видит новую версию
    .config("spark.sql.catalog.spark_catalog.cache-enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/20 13:47:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<a id="ch3"></a>
## 3. Загрузка CSV в HDFS

**Путь:** копируем CSV из папки проекта в `HDFS` → читаем его, смотрим объём и первые записи.

Spark читает данные из `HDFS`: исполнители работают на других узлах кластера и локальный диск этой машины не видят. Копируем файл один раз — если он уже в `HDFS`, команда пропускается.

In [4]:
local_csv = BASE / "data" / "US_Accidents_March23.csv"
hdfs_csv = f"{HDFS_ROOT}/data/US_Accidents_March23.csv"

!hdfs dfs -mkdir -p {HDFS_ROOT}/data
!hdfs dfs -test -e {hdfs_csv} || hdfs dfs -put {local_csv} {hdfs_csv}
!hdfs dfs -ls -h {HDFS_ROOT}/data

Found 1 items
-rw-r--r--   1 hadoop supergroup      2.8 G 2026-10-20 13:51 hdfs:///user/hadoop/SOBD-labs/data/US_Accidents_March23.csv


Файл в `HDFS`. Все столбцы читаем как строки: автоопределение типов (`inferSchema`) прошло бы по 2,8 ГБ лишний раз. Типы зададим сами в главе 6.

In [5]:
raw = spark.read.option("header", True).csv(hdfs_csv)

print(f"Строк: {raw.count():,}")
print(f"Столбцов: {len(raw.columns)}")
print(f"Разделов: {raw.rdd.getNumPartitions()}")

Строк: 7,728,394
Столбцов: 46
Разделов: 23


7,7 млн записей и 46 признаков. Spark разбил файл на разделы по блокам `HDFS` (128 МБ) и читает их параллельно.

В `show()` 46 столбцов не помещаются, поэтому выводим три записи «лёжа»: строка — признак, столбец — запись.

In [6]:
raw.limit(3).toPandas().T

,0,1,2
ID,A-1,A-2,A-3
Source,Source2,Source2,Source2
Severity,3,2,2
Start_Time,2016-02-08 05:46:00,2016-02-08 06:07:59,2016-02-08 06:49:27
End_Time,2016-02-08 11:00:00,2016-02-08 06:37:59,2016-02-08 07:19:27
Start_Lat,39.865147,39.92805900000001,39.063148
Start_Lng,-84.058723,-82.831184,-84.032608
End_Lat,None,None,None
End_Lng,None,None,None
Distance(mi),0.01,0.01,0.01


<a id="ch4"></a>
## 4. Описание признаков

**Путь:** разбираемся, что означает каждый столбец, чтобы дальше правильно выбрать типы.

| Группа | Признаки | Смысл |
| --- | --- | --- |
| Идентификация | `ID`, `Source` | номер записи и API-источник, сообщивший о ДТП |
| Тяжесть | `Severity` | влияние на движение от 1 (короткая задержка) до 4 (долгая) |
| Время | `Start_Time`, `End_Time` | начало и конец влияния ДТП на движение, местное время |
| Место | `Start_Lat`, `Start_Lng`, `End_Lat`, `End_Lng` | координаты начала и конца затронутого участка |
| Протяжённость | `Distance(mi)` | длина затронутого участка дороги, мили |
| Описание | `Description` | текст сообщения о ДТП |
| Адрес | `Street`, `City`, `County`, `State`, `Zipcode`, `Country`, `Timezone` | улица, город, округ, штат, индекс, страна, часовой пояс |
| Метеостанция | `Airport_Code`, `Weather_Timestamp` | ближайший аэропорт и время замера погоды |
| Погода | `Temperature(F)`, `Wind_Chill(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Direction`, `Wind_Speed(mph)`, `Precipitation(in)`, `Weather_Condition` | температура и ощущаемая температура (°F), влажность, давление (дюймы рт. ст.), видимость (мили), ветер, осадки (дюймы), словесное описание погоды |
| Объекты рядом | `Amenity`, `Bump`, `Crossing`, `Give_Way`, `Junction`, `No_Exit`, `Railway`, `Roundabout`, `Station`, `Stop`, `Traffic_Calming`, `Traffic_Signal`, `Turning_Loop` | есть ли рядом: инфраструктура, «лежачий полицейский», переход, знак «уступи», съезд, тупик, ж/д переезд, кольцо, остановка, знак «стоп», успокоение движения, светофор, разворотная петля |
| Время суток | `Sunrise_Sunset`, `Civil_Twilight`, `Nautical_Twilight`, `Astronomical_Twilight` | день или ночь по четырём определениям сумерек |

<a id="ch5"></a>
## 5. Имена столбцов

**Путь:** проверяем имена на неудобные символы и приводим их к единому виду.

In [7]:
[c for c in raw.columns if not c.replace("_", "").isalnum()]

['Distance(mi)',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Speed(mph)',
 'Precipitation(in)']

В восьми именах есть скобки и знак процента: в коде такие имена приходится брать в обратные кавычки, а Parquet скобки в именах не допускает.

Приводим все имена к `snake_case`: `Temperature(F)` → `temperature_f`, `Humidity(%)` → `humidity_pct`.

In [8]:
def to_snake(name):
    name = name.replace("(%)", "_pct")         # Humidity(%)  -> Humidity_pct
    name = re.sub(r"\((\w+)\)", r"_\1", name)  # Distance(mi) -> Distance_mi
    return name.lower()


df = raw.toDF(*[to_snake(c) for c in raw.columns])

pd.DataFrame({"было": raw.columns, "стало": df.columns})

,было,стало
0,ID,id
1,Source,source
2,Severity,severity
3,Start_Time,start_time
4,End_Time,end_time
5,Start_Lat,start_lat
6,Start_Lng,start_lng
7,End_Lat,end_lat
8,End_Lng,end_lng
9,Distance(mi),distance_mi


<a id="ch6"></a>
## 6. Типы данных

**Путь:** раскладываем столбцы по типам → проверяем, что значения действительно такие → приводим → убеждаемся, что ничего не потеряли.

Остальные столбцы (адрес, описание, погода словами, время суток) остаются строками.

In [9]:
int_cols = ["severity"]
double_cols = ["start_lat", "start_lng", "end_lat", "end_lng", "distance_mi",
               "temperature_f", "wind_chill_f", "humidity_pct", "pressure_in",
               "visibility_mi", "wind_speed_mph", "precipitation_in"]
time_cols = ["start_time", "end_time", "weather_timestamp"]
bool_cols = ["amenity", "bump", "crossing", "give_way", "junction", "no_exit", "railway",
             "roundabout", "station", "stop", "traffic_calming", "traffic_signal", "turning_loop"]

Перед приведением к `boolean` проверяем, какие значения встречаются во всех 13 логических столбцах.

In [10]:
df.select(F.explode(F.array(*bool_cols)).alias("value")).distinct().toPandas()

,value
0,True
1,False


Только `True` и `False` — приводим. Строки, которые не удаётся разобрать, Spark молча превращает в `NULL`, поэтому сразу после приведения считаем такие потери по каждому столбцу.

In [11]:
types = ({c: "int" for c in int_cols}
         | {c: "double" for c in double_cols}
         | {c: "timestamp" for c in time_cols}
         | {c: "boolean" for c in bool_cols})

typed = df
for c, t in types.items():
    typed = typed.withColumn(c, F.col(c).cast(t))

# значение было в CSV, но после приведения стало NULL
lost = df.select([
    F.sum((F.col(c).isNotNull() & F.col(c).cast(t).isNull()).cast("int")).alias(c)
    for c, t in types.items()
]).first().asDict()

{c: n for c, n in lost.items() if n > 0}

{}

Потерь нет: все 29 столбцов приведены без единого испорченного значения. Итоговая схема:

In [12]:
pd.DataFrame(typed.dtypes, columns=["столбец", "тип"])

,столбец,тип
0,id,string
1,source,string
2,severity,int
3,start_time,timestamp
4,end_time,timestamp
5,start_lat,double
6,start_lng,double
7,end_lat,double
8,end_lng,double
9,distance_mi,double


<a id="ch7"></a>
## 7. Сохранение в Iceberg

**Путь:** создаём базу данных → записываем таблицу → проверяем, что записи на месте, и смотрим, сколько места она заняла.

In [13]:
database = "sobd_accidents"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database}")
spark.catalog.setCurrentDatabase(database)

typed.writeTo("us_accidents_raw").using("iceberg").createOrReplace()

Проверяем: таблица видна в каталоге и число строк совпадает с исходным CSV.

In [14]:
print([table.name for table in spark.catalog.listTables()])

print(f"Строк в таблице: {spark.table('us_accidents_raw').count():,}")

['us_accidents_raw']


Строк в таблице: 7,728,394


Все 7,7 млн записей на месте. Сравним объём таблицы с исходным CSV на 2,8 ГБ.

Объём берём из метаданных Iceberg: служебная таблица `files` перечисляет файлы данных текущей версии таблицы.

In [15]:
size = spark.sql(f"SELECT COUNT(*) AS files, SUM(file_size_in_bytes) AS bytes FROM {database}.us_accidents_raw.files").first()
print(f"Файлов данных: {size['files']}, объём: {size['bytes'] / 2**20:.0f} МБ")

Файлов данных: 23, объём: 493 МБ


Таблица заняла около 490 МБ — почти в шесть раз меньше CSV: колоночный формат Parquet хорошо сжимается, а читать из него можно только нужные столбцы.

Iceberg хранит историю изменений таблицы в виде снимков (snapshots): каждый запуск ноутбука заменяет таблицу и добавляет новый снимок, а старые остаются — к любому из них можно вернуться (time travel). Поэтому каталог таблицы в `HDFS` после перезапусков больше, чем её текущая версия.

In [16]:
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {database}.us_accidents_raw.snapshots").toPandas()

,committed_at,snapshot_id,operation
0,2026-10-20 13:54:41.832,3422222979353566162,append


<a id="ch8"></a>
## 8. Выводы

* CSV на 2,8 ГБ прочитан из `HDFS` параллельно по разделам: 7,7 млн записей, 46 признаков.
* Имена столбцов приведены к `snake_case`, 29 столбцов получили числовые, временные и логические типы без потерь.
* Данные сохранены в таблицу `sobd_accidents.us_accidents_raw` формата `Apache Iceberg`; объём уменьшился почти в шесть раз — с 2,8 ГБ до ~490 МБ.

Дальше — часть 2: разведочный анализ, очистка и сохранение очищенной таблицы.

In [17]:
spark.stop()